In [ ]:
import onep
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt
# from cebra import CEBRA, plot_loss, plot_embedding
from scipy.signal import resample
import statsmodels.api as sm

In [ ]:
traces_4 = onep.InscopixProcessing(session="recall", animal="astro4", data_directory="/home/ryansenne/Data/RLS_Team_Data/dCA1")
traces_4.read_inscopix()

traces_5 = onep.InscopixProcessing(session="recall", animal="astro5", data_directory="/home/ryansenne/Data/RLS_Team_Data/dCA1")
traces_5.read_inscopix()

traces_6 = onep.InscopixProcessing(session="recall", animal="astro6", data_directory="/home/ryansenne/Data/RLS_Team_Data/dCA1")
traces_6.read_inscopix()

traces_3 = onep.InscopixProcessing(session="recall", animal="astro3", data_directory="/home/ryansenne/Data/RLS_Team_Data/dCA1")
traces_3.read_inscopix()

dca1_group = onep.dCA1Group(traces_3, traces_4, traces_5, traces_6)
dca1_group.preprocess()

In [ ]:
modely = sm.OLS(resample(traces_5.dlc_df["centroid"].y, len(traces_5.Timestamps)), sm.add_constant(traces_5.accepted_traces)).fit()
modelx = sm.OLS(resample(traces_5.dlc_df["centroid"].x, len(traces_5.Timestamps)), sm.add_constant(traces_5.accepted_traces)).fit()


In [ ]:
print(modelx.summary())

In [ ]:
fig, ax = plt.subplots()
ax.plot(modely.predict())
ax.plot(resample(traces_5.dlc_df["centroid"].y, len(traces_4.Timestamps)))

In [ ]:
fig, ax = plt.subplots()
# ax.plot(traces_4.dlc_df["centroid"].x, traces_4.dlc_df["centroid"].y)
ax.plot(modelx.predict(), modely.predict())

In [ ]:
# import autograd.numpy as np
import autograd.numpy.random as npr
npr.seed(0)

import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

%matplotlib inline

import seaborn as sns

sns.set_style("white")
sns.set_context("talk")

color_names = ["windows blue",
               "red",
               "amber",
               "faded green",
               "dusty purple",
               "orange",
               "clay",
               "pink",
               "greyish",
               "mint",
               "cyan",
               "steel blue",
               "forest green",
               "pastel purple",
               "salmon",
               "dark brown"]

colors = sns.xkcd_palette(color_names)
cmap = ListedColormap(colors)

import ssm
from ssm.util import random_rotation, find_permutation
from ssm.plots import plot_dynamics_2d

save_figures = False

In [ ]:
ss = StandardScaler()
scaled_traces = ss.fit_transform(traces_4.accepted_traces)

In [ ]:
inds = scaled_traces.shape[0]
obs_dim = scaled_traces.shape[1]
discrete_states = 10
cont_dim = 32

fc_rslds = ssm.SLDS(obs_dim,
                     discrete_states,
                     cont_dim,
                     emissions="gaussian_orthog",
                     transitions="recurrent"
                     )

q_mf_elbos, q_lem = fc_rslds.fit(scaled_traces, 
                    method="laplace_em", 
                    variational_posterior="structured_meanfield", 
                    num_iters=100, alpha=0.0)

xhat_lem = q_lem.mean_continuous_states[0]
zhat_lem = fc_rslds.most_likely_states(xhat_lem, scaled_traces)

In [ ]:
plt.plot(q_mf_elbos)

In [ ]:
plt.plot(xhat_lem)

In [ ]:
import cebra

cebra_model = cebra.CEBRA(
    batch_size=None,
    model_architecture="offset10-model",
    temperature_mode="auto",
    max_iterations=10000,
    output_dimension=3
)

cebra_model.fit(traces_5.accepted_traces, resample(traces_5.dlc_df["centroid"], 3602))

In [ ]:
embedding = cebra_model.transform(traces_5.accepted_traces)
cebra.plot_embedding(embedding,embedding_labels=None,  cmap="cool", figsize=(10,10))

In [ ]:
cebra.plot_loss(cebra_model)